# Addition matrix 2 · The law of the factor search, and the address of the final answer
**Engine:** `ValaQuenta.modules.addition_matrix` · confidence floor ESTABLISHED.

Digit-by-digit factor reconstruction keeps exactly the pairs (p, q) mod 10^(c+1) with p·q ≡ N there. The count has a closed form, a product over the primes 2 and 5. Nothing here is a faster way to factor: the live-state count is the quantity measured.

In [1]:
from ValaQuenta.modules.addition_matrix import maths as M
for N, n in ((3233, 2), (14351, 3), (1022117, 4), (420, 2), (850, 2), (5100, 2)):
    live = M.live_state_counts(N, n)
    print(f'N={N:>8} v2={M.valuation(N,2)} v5={M.valuation(N,5)}  live per column {live}\n{"":>31}closed form      {[M.solution_count_closed(N, c+1) for c in range(n)]}')

N=    3233 v2=0 v5=0  live per column [4, 40, 6, 2]
                               closed form      [4, 40]
N=   14351 v2=0 v5=0  live per column [4, 40, 400, 48, 4, 2]
                               closed form      [4, 40, 400]
N= 1022117 v2=0 v5=0  live per column [4, 40, 400, 4000, 406, 38, 8, 2]
                               closed form      [4, 40, 400, 4000]
N=     420 v2=2 v5=1  live per column [27, 320, 32, 16]
                               closed form      [27, 320]
N=     850 v2=1 v5=2  live per column [27, 260, 26, 6]
                               closed form      [27, 260]
N=    5100 v2=2 v5=2  live per column [27, 520, 32, 4]
                               closed form      [27, 520]


Columns *below n* equal the closed form exactly; from column n on there are no new digits, the size of the factors enters, and the count collapses (here to 2: the pair and its swap).

In [2]:
bad = [(N, k) for k in (1, 2, 3) for N in range(0, 2*10**k + 50) if M.solution_count_closed(N, k) != M.solution_count_brute(N, k)]
print('closed form vs enumeration, N in [0, 2*10^k+50) for k = 1, 2, 3 — mismatches:', len(bad))

closed form vs enumeration, N in [0, 2*10^k+50) for k = 1, 2, 3 — mismatches: 0


**Unit-group torsors.** With both valuations below k the solution set is (v₂+1)(v₅+1) orbits of (ℤ/10ᵏ)ˣ, which has order 4·10^(k−1): this is the ×10 per digit. Every solution has exactly ten lifts.

In [3]:
print([M.unit_group_order(k) for k in (1, 2, 3, 4)])
print(M.torsor_orbits(14351, 3), M.torsor_orbits(420, 3), M.lift_branching(14351, 2))

[4, 40, 400, 4000]
1 6 {10: 40}


**The address.** For a unit solution mod 10ⁿ, K = (p′q′ − N)/10ⁿ is an integer. The true factorisation is exactly K = 0: it has an address. It is not isolated: other solutions have K = ±1, ±2, … Both are computed below for products of 3-digit primes.

In [4]:
import random
rng = random.Random(3)
P3 = [x for x in range(101, 999) if all(x % d for d in range(2, int(x**.5)+1))]
rows = []
for _ in range(30):
    p, q = rng.sample(P3, 2)
    pr = M.address_profile(p*q, 3)
    rows.append((pr['solutions'], pr['zero_overflow'], pr['smallest_nonzero_K'], pr['median_nonzero_K']))
print('solutions per N:', {r[0] for r in rows}, '| K=0 solutions:', {r[1] for r in rows})
print('smallest nonzero |K|: min', min(r[2] for r in rows), 'max', max(r[2] for r in rows), '| median |K|', sorted(r[3] for r in rows)[len(rows)//2])

solutions per N: {400} | K=0 solutions: {2}
smallest nonzero |K|: min 1 max 14 | median |K| 171.0


**What it does not do.** A K = 0 test needs every solution mod 10ⁿ, which is the 4·10^(n−1)-element set; no coordinate of the unit group singles the true pair out in advance. The module reports the address and its neighbours; it is not a factoring method.